# Clustering Jerarquico — Introduccion con datos sinteticos

---

**Autor:** Borja Mora Méndez
**Contacto:** [borja.mora.mendez@gmail.com](mailto:borja.mora.mendez@gmail.com) · [LinkedIn](https://www.linkedin.com/in/borja-mora-mendez/)
**Repositorio:** [Data Analytics Portfolio](https://github.com/BORJAMOME/Data-Analytics-Portfolio)
**Categoría:** Machine Learning · No Supervisado · Clustering · Jerárquico

---

**Objetivo:** Entender los fundamentos del clustering jerarquico — dendrogramas, metodos de enlace y como decidir el numero optimo de clusters — usando un dataset sintetico de comportamiento de usuarios en e-commerce.

**Contexto de negocio:** Un equipo de marketing digital necesita segmentar usuarios segun su actividad (clicks y compras) para personalizar campanas. No tienen etiquetas previas — necesitan descubrir los grupos de forma no supervisada.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist
from sklearn.preprocessing import StandardScaler

sns.set_style("whitegrid")
np.random.seed(42)

# Estilo visual — sistema de color validado (consejo UX/UI Data)
BACKGROUND    = '#fbfbfb'
PURPLE        = '#7a7bff'   # único color de énfasis / serie única en scatter y líneas (1 por gráfico)
PURPLE_LIGHT  = '#9b9cff'   # EDA de una sola serie (histogramas independientes)
POSITIVE      = '#6b8158'   # exclusivo signo positivo
NEGATIVE      = '#c34031'   # exclusivo signo negativo
NEUTRAL_BAR   = '#d9d9d9'   # barras/áreas de contexto (siempre con etiqueta de valor)
NEUTRAL_LINE  = '#8f8c9e'   # líneas de contexto
CONTEXT_LINES = [NEUTRAL_LINE, '#a89a8a', '#7d94a8']   # gama fija para 2+ líneas de contexto
INK           = '#111111'
MUTED         = '#707070'

# Paleta categórica para identidad de cluster — validada (ΔE OKLab, simulación CVD) para
# pares adyacentes (barras, líneas, enlaces de dendrograma). En scatter/PCA con 4+ clusters
# el color por sí solo no basta para daltonismo severo: por eso cada cluster lleva también
# una forma de marcador distinta (CLUSTER_MARKERS) — nunca dependas solo del color.
CLUSTER_PALETTE = ['#7a7bff', '#eb6834', '#1baf7a', '#e34948', '#eda100', '#e87ba4', '#008300']
CLUSTER_MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X']

DIVERGING_CMAP = LinearSegmentedColormap.from_list(
    "borja_diverging", ["#c34031", "#e0a89f", "#f0ede8", "#b7c2a9", "#6b8158"]
)
SEQUENTIAL_GREEN = LinearSegmentedColormap.from_list(
    "borja_sequential", [BACKGROUND, POSITIVE]
)

def color_annotations(ax, values, threshold, dark="#ffffff", light=INK):
    """Recolorea el texto de un heatmap celda a celda según su magnitud."""
    for text, value in zip(ax.texts, np.asarray(values).flatten()):
        text.set_color(dark if abs(value) >= threshold else light)

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'figure.dpi': 100,
    'figure.facecolor': BACKGROUND,
    'axes.facecolor': BACKGROUND,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': MUTED,
    'axes.labelcolor': INK,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.titlecolor': INK,
    'xtick.color': MUTED,
    'ytick.color': MUTED,
    'font.family': 'sans-serif',
    'font.size': 10,
    'grid.color': '#f0f0f0',
    'grid.linewidth': 0.5,
})

print("Librerias cargadas correctamente")

## 1. Generacion del dataset sintetico

Simulamos 30 usuarios de e-commerce con dos metricas de comportamiento: numero de clicks por sesion y numero de compras mensuales. Creamos 3 grupos naturales que el algoritmo deberia descubrir.

In [2]:
# Generar 3 grupos naturales de usuarios
grupo_casual = np.column_stack([
    np.random.uniform(1, 4, 10),   # pocos clicks
    np.random.uniform(0, 2, 10)    # pocas compras
])
grupo_activo = np.column_stack([
    np.random.uniform(5, 8, 10),   # clicks medios
    np.random.uniform(3, 6, 10)    # compras medias
])
grupo_premium = np.column_stack([
    np.random.uniform(8, 12, 10),  # muchos clicks
    np.random.uniform(7, 12, 10)   # muchas compras
])

datos = np.vstack([grupo_casual, grupo_activo, grupo_premium])
df = pd.DataFrame(datos, columns=["Clicks_Sesion", "Compras_Mes"])

print(f"Dataset: {df.shape[0]} usuarios, {df.shape[1]} variables")
print(f"\nEstadisticas descriptivas:")
display(df.describe().round(2))

Dataset: 30 usuarios, 2 variables

Estadisticas descriptivas:


,Clicks_Sesion,Compras_Mes
count,30.00,30.00
mean,6.13,5.07
std,3.06,3.99
min,1.17,0.04
25%,3.30,1.20
50%,6.23,4.57
75%,8.68,7.85
max,11.64,11.85


## 2. Exploracion visual

Antes de aplicar cualquier algoritmo, visualizamos los datos para ver si hay agrupaciones naturales.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df["Clicks_Sesion"], df["Compras_Mes"], color=PURPLE, s=60, alpha=0.7, edgecolors="k", linewidths=0.5)
for i, row in df.iterrows():
    ax.annotate(str(i), (row["Clicks_Sesion"], row["Compras_Mes"]), fontsize=7, ha="center", va="bottom")
ax.set_xlabel("Clicks por sesion")
ax.set_ylabel("Compras mensuales")
ax.set_title("Distribucion de usuarios — datos sin etiquetar")
plt.tight_layout()
plt.show()

## 3. Normalizacion

Las variables tienen escalas similares en este caso, pero normalizar es buena practica — evita que una variable domine la distancia solo por tener mayor rango.

In [4]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

print(f"Media post-escalado:  {X_scaled.mean(axis=0).round(4)}")
print(f"Desv. post-escalado:  {X_scaled.std(axis=0).round(4)}")

Media post-escalado:  [ 0. -0.]
Desv. post-escalado:  [1. 1.]


## 4. Dendrograma y metodos de enlace

El dendrograma es la herramienta visual clave del clustering jerarquico. Muestra como se van fusionando los puntos y a que distancia.

Comparamos 4 metodos de enlace para ver como cambia la estructura:
- **Ward:** minimiza la varianza intra-cluster (tiende a crear clusters compactos y equilibrados)
- **Complete:** usa la distancia maxima entre puntos de dos clusters
- **Average:** usa la distancia media entre todos los pares
- **Single:** usa la distancia minima (sensible a efecto cadena)

In [ ]:
metodos = ["ward", "complete", "average", "single"]

from scipy.cluster.hierarchy import set_link_color_palette
set_link_color_palette(CLUSTER_PALETTE)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, metodo in enumerate(metodos):
    Z = linkage(X_scaled, method=metodo, metric="euclidean")
    dendrogram(Z, ax=axes[i], leaf_rotation=90, leaf_font_size=8, above_threshold_color=INK)
    axes[i].set_title(f"Metodo: {metodo.capitalize()}", fontsize=13, fontweight="bold")
    axes[i].set_xlabel("Indice del usuario")
    axes[i].set_ylabel("Distancia")
    axes[i].axhline(y=4 if metodo == "ward" else 3, color=INK, linestyle="--", alpha=0.5, label="Corte sugerido")
    axes[i].legend(fontsize=8)

plt.suptitle("Comparacion de metodos de enlace", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 5. Metodo del codo para elegir k

Analizamos las distancias de fusion para identificar el punto donde se produce el mayor salto — ese salto indica el numero optimo de clusters.

In [ ]:
Z_ward = linkage(X_scaled, method="ward", metric="euclidean")

distancias = Z_ward[:, 2]
ultimas = distancias[-10:][::-1]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(ultimas) + 1), ultimas, marker="o", linewidth=2, color=PURPLE)
ax.set_xlabel("Numero de clusters")
ax.set_ylabel("Distancia de fusion")
ax.set_title("Metodo del codo — Clustering Jerarquico (Ward)")
ax.axvline(x=3, color=INK, linestyle="--", alpha=0.6, label="k=3 (codo)")
ax.legend()
ax.set_xticks(range(1, len(ultimas) + 1))
plt.tight_layout()
plt.show()

print("El mayor salto se produce entre k=3 y k=2, confirmando 3 clusters.")

## 6. Asignacion final y visualizacion

Cortamos el dendrograma en k=3 y visualizamos los clusters resultantes.

In [ ]:
# Cortar en 3 clusters
labels = fcluster(Z_ward, t=3, criterion="maxclust")
df["Cluster"] = labels

# Visualizar
colores = {1: CLUSTER_PALETTE[0], 2: CLUSTER_PALETTE[1], 3: CLUSTER_PALETTE[2]}
nombres = {1: "Casual", 2: "Activo", 3: "Premium"}

fig, ax = plt.subplots(figsize=(8, 6))
for cl in sorted(df["Cluster"].unique()):
    subset = df[df["Cluster"] == cl]
    ax.scatter(subset["Clicks_Sesion"], subset["Compras_Mes"], 
               s=70, label=f"Cluster {cl}: {nombres[cl]}", 
               color=colores[cl], marker=CLUSTER_MARKERS[cl - 1], edgecolors="k", linewidths=0.5)

ax.set_xlabel("Clicks por sesion")
ax.set_ylabel("Compras mensuales")
ax.set_title("Segmentacion de usuarios — 3 clusters (Ward)")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Perfil de cada cluster

In [8]:
perfil = df.groupby("Cluster")[["Clicks_Sesion", "Compras_Mes"]].agg(["mean", "std", "count"])
perfil.columns = ["_".join(col) for col in perfil.columns]
perfil = perfil.round(2)

print("Perfil de cada cluster:")
display(perfil)

print("\nInterpretacion:")
for cl in sorted(df["Cluster"].unique()):
    sub = df[df["Cluster"] == cl]
    print(f"  Cluster {cl} ({nombres[cl]}): {len(sub)} usuarios, "
          f"clicks={sub['Clicks_Sesion'].mean():.1f}, compras={sub['Compras_Mes'].mean():.1f}")

Perfil de cada cluster:


,Clicks_Sesion_mean,Clicks_Sesion_std,Clicks_Sesion_count,Compras_Mes_mean,Compras_Mes_std,Compras_Mes_count
Cluster,,,,,,
1,9.62,1.08,10,9.88,1.88,10
2,6.20,0.70,10,4.53,1.03,10
3,2.56,0.95,10,0.79,0.60,10



Interpretacion:
  Cluster 1 (Casual): 10 usuarios, clicks=9.6, compras=9.9
  Cluster 2 (Activo): 10 usuarios, clicks=6.2, compras=4.5
  Cluster 3 (Premium): 10 usuarios, clicks=2.6, compras=0.8


## 8. Conclusiones y recomendaciones de negocio

**Hallazgos:**
- El clustering jerarquico con metodo Ward identifica correctamente los 3 segmentos naturales de usuarios.
- El dendrograma proporciona una vista completa de la estructura de agrupacion a multiples niveles de granularidad.
- El metodo del codo confirma k=3 como numero optimo.

**Recomendaciones para marketing:**
- **Casual (bajo engagement):** Campanas de reactivacion, descuentos de primer compra.
- **Activo (engagement medio):** Programas de fidelizacion, cross-selling.
- **Premium (alto engagement):** Acceso anticipado a productos, programa VIP.

**Limitaciones:**
- Dataset sintetico con separacion clara — en datos reales los limites entre clusters son mas difusos.
- Solo 2 variables — en produccion se usarian mas dimensiones (requiere PCA previo).
- El clustering jerarquico no escala bien a datasets grandes (complejidad O(n^2) en memoria).